In [1]:
!nvidia-smi

Wed Jul  8 06:23:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount("/content/drive")
import json
from datasets import load_from_disk
BASE = "/content/drive/MyDrive/ner" # adjust to your folder
tok_ds = load_from_disk(f"{BASE}/uzbek_ner_tokenized")
label_list = json.load(open(f"{BASE}/label_list.json"))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print(tok_ds)
print(label_list)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15687
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1961
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1961
    })
})
['O', 'B-PERSON', 'I-PERSON', 'B-ORG', 'I-ORG', 'B-GPE', 'I-GPE', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE']


In [8]:
from transformers import AutoTokenizer
MODEL = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(MODEL)

In [4]:
!pip install -q seqeval

In [5]:
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=2)
    true_preds, true_labels = [], []
    for pr, la in zip(preds, labels):
        tp = [id2label[int(q)] for q, l in zip(pr, la) if l != -100]
        tl = [id2label[int(l)] for q, l in zip(pr, la) if l != -100]
        true_preds.append(tp)
        true_labels.append(tl)
    r = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": r["overall_precision"],
        "recall": r["overall_recall"],
        "f1": r["overall_f1"],
        "accuracy": r["overall_accuracy"],
    }

In [6]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


277.461515 M parameters


In [7]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="xlmr-uzner-baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.06,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    seed=42,
    report_to="none",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
from transformers import Trainer, DataCollatorForTokenClassification

collator = DataCollatorForTokenClassification(tok)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_ds["train"],
    eval_dataset=tok_ds["validation"],
    data_collator=collator,
    processing_class=tok,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.168311,0.160325,0.648582,0.699126,0.672906,0.939480
2,0.148872,0.152812,0.651828,0.728297,0.687944,0.941747
3,0.133570,0.149840,0.668389,0.741938,0.703246,0.942868
4,0.123794,0.150230,0.675026,0.740049,0.706043,0.943027


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.168311,0.160325,0.648582,0.699126,0.672906,0.939480
2,0.148872,0.152812,0.651828,0.728297,0.687944,0.941747
3,0.133570,0.149840,0.668389,0.741938,0.703246,0.942868
4,0.123794,0.150230,0.675026,0.740049,0.706043,0.943027


TrainOutput(global_step=3924, training_loss=0.1814669049843371, metrics={'train_runtime': 1367.6713, 'train_samples_per_second': 45.879, 'train_steps_per_second': 2.869, 'total_flos': 8125196351852490.0, 'train_loss': 0.1814669049843371, 'epoch': 4.0})

In [10]:
from seqeval.metrics import classification_report

pred = trainer.predict(tok_ds["validation"])
preds = np.argmax(pred.predictions, axis=2)
labels = pred.label_ids

val_pred_seqs, val_gold_seqs = [], []
for pr, la in zip(preds, labels):
    val_pred_seqs.append([id2label[int(q)] for q, l in zip(pr, la) if l != -100])
    val_gold_seqs.append([id2label[int(l)] for q, l in zip(pr, la) if l != -100])

print(classification_report(val_gold_seqs, val_pred_seqs, digits=3))
print("Overall metrics:", trainer.evaluate())

              precision    recall  f1-score   support

        DATE      0.399     0.515     0.450      1020
         GPE      0.716     0.834     0.771      4914
         LOC      0.501     0.460     0.479      2141
         ORG      0.635     0.717     0.673      3249
      PERSON      0.881     0.889     0.885      2971

   micro avg      0.675     0.740     0.706     14295
   macro avg      0.626     0.683     0.652     14295
weighted avg      0.677     0.740     0.706     14295



Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.123794,0.150230,4,0.675026,0.740049,0.706043,0.943027


Overall metrics: {'eval_loss': 0.15022975206375122, 'eval_precision': 0.6750255232261357, 'eval_recall': 0.7400489681706891, 'eval_f1': 0.7060433143124104, 'eval_accuracy': 0.9430272911846641}


In [11]:
def merge_place(seqs):
    return [[t.replace("-GPE", "-PLACE").replace("-LOC", "-PLACE") for t in s]
            for s in seqs]

print(classification_report(merge_place(val_gold_seqs),
                            merge_place(val_pred_seqs), digits=3))

              precision    recall  f1-score   support

        DATE      0.399     0.515     0.450      1020
         ORG      0.635     0.717     0.673      3249
      PERSON      0.881     0.889     0.885      2971
       PLACE      0.789     0.852     0.820      7055

   micro avg      0.738     0.805     0.770     14295
   macro avg      0.676     0.743     0.707     14295
weighted avg      0.745     0.805     0.773     14295



In [12]:
def drop_date(seqs):
    return [["O" if t.endswith("DATE") else t for t in s] for s in seqs]

print(classification_report(drop_date(val_gold_seqs),
                            drop_date(val_pred_seqs), digits=3))

              precision    recall  f1-score   support

         GPE      0.716     0.834     0.771      4914
         LOC      0.501     0.460     0.479      2141
         ORG      0.635     0.717     0.673      3249
      PERSON      0.881     0.889     0.885      2971

   micro avg      0.700     0.757     0.728     13275
   macro avg      0.683     0.725     0.702     13275
weighted avg      0.698     0.757     0.725     13275



In [13]:
from collections import Counter

conf = Counter()
for g_seq, p_seq in zip(val_gold_seqs, val_pred_seqs):
    for g, p in zip(g_seq, p_seq):
        if g != "O" and p != "O" and g[2:] != p[2:]:
            conf[(g[2:], p[2:])] += 1

for pair, c in conf.most_common(8):
    print(f"{pair[0]:7} -> {pair[1]:7} {c:5}")

LOC     -> GPE       730
GPE     -> LOC       441
ORG     -> GPE       336
GPE     -> ORG       307
LOC     -> ORG       106
ORG     -> LOC        29
PERSON  -> LOC        19
PERSON  -> ORG        18


In [14]:
print(classification_report(merge_place(drop_date(val_gold_seqs)),
                            merge_place(drop_date(val_pred_seqs)), digits=3))

              precision    recall  f1-score   support

         ORG      0.635     0.717     0.673      3249
      PERSON      0.881     0.889     0.885      2971
       PLACE      0.789     0.852     0.820      7055

   micro avg      0.769     0.827     0.797     13275
   macro avg      0.768     0.819     0.793     13275
weighted avg      0.772     0.827     0.798     13275



In [15]:
splits = load_from_disk(f"{BASE}/uzbek_ner_splits")
val_words = splits["validation"]

import random
random.seed(42)

shown = 0
for i in random.sample(range(len(val_words)), 300):
    toks = val_words[i]["tokens"]
    gold = val_words[i]["tags"]
    pl = val_pred_seqs[i]
    n = min(len(gold), len(pl))          # guard for any truncated row
    if any(g != p for g, p in zip(gold[:n], pl[:n])):
        print(f"===== validation row {i} =====")
        for w, g, p in zip(toks[:n], gold[:n], pl[:n]):
            mark = "" if g == p else f"   <-- pred: {p}"
            print(f"{w:22} {g:10}{mark}")
        print()
        shown += 1
    if shown == 3:
        break

===== validation row 1309 =====
Angliya                O         
Premer-Ligasi          O         
6-turidan              O         
o'rin                  O         
olgan                  O         
bir                    O         
o'yin                  O         
yakunlandi.            O         
«Tottenhem»            B-ORG     
safarda                O         
«Brayton»              B-GPE        <-- pred: B-ORG
bilan                  O         
bellashdi              O         
va                     O         
2:1                    O         
hisobidagi             O         
g'alabani              O         
qo'lga                 O         
kiritdi.               O         
Mehmonlarning          O         
gollari                O         
Keyn                   B-PERSON  
va                     O         
Lamela                 B-PERSON  
hisobiga               O         
yozildi.               O         
Natijada,              O         
«Tottenhem»            B-ORG    

In [16]:
import json, os
import pandas as pd
from seqeval.metrics import f1_score

SAVE = f"{BASE}/baseline_xlmr"          # Drive path — a ~1.1 GB folder
trainer.save_model(SAVE)
tok.save_pretrained(SAVE)
print("saved:", sorted(os.listdir(SAVE)))

json.dump(trainer.state.log_history,
          open(f"{BASE}/baseline_train_log.json", "w"))

best = trainer.evaluate()

views = {
    "f1_5class": round(f1_score(val_gold_seqs, val_pred_seqs), 4),
    "f1_place":  round(f1_score(merge_place(val_gold_seqs),
                                merge_place(val_pred_seqs)), 4),
    "f1_nodate": round(f1_score(drop_date(val_gold_seqs),
                                drop_date(val_pred_seqs)), 4),
    "f1_core":   round(f1_score(merge_place(drop_date(val_gold_seqs)),
                                merge_place(drop_date(val_pred_seqs))), 4),
}

ledger = pd.DataFrame([{
    "run": "baseline",
    "model": MODEL,
    "max_length": 256,        # <-- REPLACE with YOUR Phase 4 audit value (the backfill!)
    "lr": 2e-5,
    "epochs": 4,
    "eff_batch": 16,
    "seed": 42,
    "val_precision": round(best["eval_precision"], 4),
    "val_recall":    round(best["eval_recall"], 4),
    **views,
}])
ledger.to_csv(f"{BASE}/runs_ledger.csv", index=False)
print(ledger.to_string(index=False))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.123794,0.150230,4,0.675026,0.740049,0.706043,0.943027


     run            model  max_length      lr  epochs  eff_batch  seed  val_precision  val_recall  f1_5class  f1_place  f1_nodate  f1_core
baseline xlm-roberta-base         256 0.00002       4         16    42          0.675        0.74      0.706    0.7699     0.7277    0.797
